In [ ]:
# Mount google drive to current runtime session to access required files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import Libraries
import shutil
import pandas as pd
import numpy as np
import os
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import itertools
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
import time

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
True
NVIDIA A100-SXM4-40GB


In [ ]:
# Import spectrogram zip to local storage for efficient loading

shutil.copy('/content/drive/MyDrive/DSAN6600 Project/piano_roll.zip',
            '/content/piano_roll.zip')

'/content/piano_roll.zip'

In [ ]:
# Unzip spectrogram file to extract all 150,000 npy files
!unzip -q piano_roll.zip -d /content/piano_roll

In [ ]:
# Size of spectrogram folder and number of npy files
!du -sh /content/piano_roll/piano_roll/*
!find /content/piano_roll/piano_roll/ -type f | wc -l

/bin/bash: line 1: /usr/bin/du: Argument list too long
150014


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DSAN6600 Project/lmd_full_metadata.csv')

# Filter out missing keys
df = df[df['key_name'].notna() & (df['key_name'] != '')]

# Filter major keys only
df = df[df['key_name'].str.contains('major')]

print(f"Total usable samples: {len(df)}")
print(f"Key distribution:\n{df['key_name'].value_counts()}")

Total usable samples: 93478
Key distribution:
key_name
C major     59720
G major      6836
F major      5821
D major      4520
A# major     3707
D# major     3560
A major      3025
E major      2291
G# major     1807
C# major     1050
B major       618
F# major      523
Name: count, dtype: int64


In [ ]:
# Cap C major at 4500
df_c = df[df['key_name'] == 'C major'].sample(4500, random_state=42)
df_others = df[df['key_name'] != 'C major']

# Combine and drop keys with less than 1000 samples
df_balanced = pd.concat([df_c, df_others])
df_balanced = df_balanced.groupby('key_name').filter(lambda x: len(x) >= 1000)

print(df_balanced['key_name'].value_counts())
print(f"Total samples: {len(df_balanced)}")
print(f"Total classes: {df_balanced['key_name'].nunique()}")

key_name
G major     6836
F major     5821
D major     4520
C major     4500
A# major    3707
D# major    3560
A major     3025
E major     2291
G# major    1807
C# major    1050
Name: count, dtype: int64
Total samples: 37117
Total classes: 10


In [ ]:
# Convert keys to integers for labeling purposes
key_to_idx = {key: idx for idx, key in enumerate(sorted(df_balanced['key_name'].unique()))}
idx_to_key = {idx: key for key, idx in key_to_idx.items()}

print(key_to_idx)
# {'A major': 0, 'A# major': 1, 'C major': 2, ...}

df_balanced['label'] = df_balanced['key_name'].map(key_to_idx)

df_balanced.head()

{'A major': 0, 'A# major': 1, 'C major': 2, 'C# major': 3, 'D major': 4, 'D# major': 5, 'E major': 6, 'F major': 7, 'G major': 8, 'G# major': 9}


,file,filepath,duration_s,resolution,n_instruments,n_notes,n_tempo_changes,initial_bpm,n_key_changes,initial_key,n_time_sig_changes,initial_time_sig,key_name,label
70182,2c877e367d9ecae70c59f9a1fafff7b6.mid,../data/raw_data/lmd_full/2/2c877e367d9ecae70c...,17.35,384,4,191,1,83.0,1,0.0,1,4/4,C major,2
147776,5c513934243de942f6b27b4383403f1f.mid,../data/raw_data/lmd_full/5/5c513934243de942f6...,203.91,480,15,6463,1,133.0,1,0.0,1,4/4,C major,2
132351,19684ed364581e059e7645042b607c86.mid,../data/raw_data/lmd_full/1/19684ed364581e059e...,359.91,240,11,5510,1,80.0,1,0.0,1,4/4,C major,2
88719,065ba7baee7cda0903cceb2c54404395.mid,../data/raw_data/lmd_full/0/065ba7baee7cda0903...,189.71,480,7,3059,2,70.0,1,0.0,1,4/4,C major,2
82806,ba68ba35a4bf1a58dc2619ac72509c8f.mid,../data/raw_data/lmd_full/b/ba68ba35a4bf1a58dc...,60.78,480,1,995,1,165.0,1,0.0,1,4/4,C major,2


In [ ]:
df_balanced['npy_file'] = df_balanced['file'].str.replace("mid","npy")

# Build full file paths
data_dir = '/content/piano_roll/piano_roll/'
df_balanced['file_path'] = data_dir + df_balanced['npy_file']

In [ ]:
sample = df_balanced['file_path'].iloc[1]
print(f"Example path: {sample}")
print(f"File exists: {os.path.exists(sample)}")

Example path: /content/piano_roll/piano_roll/5c513934243de942f6b27b4383403f1f.npy
File exists: True


In [ ]:
# Check which files actually exist
df_balanced['file_exists'] = df_balanced['file_path'].apply(os.path.exists)

print(f"Files found:   {df_balanced['file_exists'].sum()}")
print(f"Files missing: {(~df_balanced['file_exists']).sum()}")

# Keep only existing files
df_balanced = df_balanced[df_balanced['file_exists']]

df_balanced = df_balanced[df_balanced['file_path'].apply(
    lambda p: np.load(p).shape == (128, 468)
)]

print(f"Remaining samples after removing different shapes: {len(df_balanced)}")


NUM_CLASSES = df_balanced['key_name'].nunique()
print(f"Number of classes: {NUM_CLASSES}")

Files found:   36112
Files missing: 0
Remaining samples after removing different shapes: 36112
Number of classes: 10


In [ ]:
sample = np.load('/content/piano_roll/piano_roll/deeb8ab7a04e5360231fb5019d847b1a.npy')
print(f"Shape: {sample.shape}")
print(f"Dtype: {sample.dtype}")
print(f"Min: {sample.min():.4f}, Max: {sample.max():.4f}")

Shape: (128, 468)
Dtype: uint8
Min: 0.0000, Max: 114.0000


In [ ]:
class PianoRollDataset(Dataset):
    def __init__(self, file_paths, labels):
        self.file_paths = file_paths
        self.labels = labels

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        x = np.load(self.file_paths[idx]).astype(np.float32)
        # Piano rolls are binary/velocity values in [0, 127] — normalize to [0, 1]
        x = x / 127.0
        x = x[np.newaxis, :, :]              # add channel dim → (1, 128, 468)
        return torch.tensor(x), torch.tensor(self.labels[idx])

In [ ]:
file_paths = df_balanced['file_path'].tolist()
labels = df_balanced['label'].tolist()

# Split into train/val

X_train, X_val, y_train, y_val = train_test_split(
    file_paths, labels,
    test_size=0.2,
    random_state=42,
    stratify=labels  # ensures balanced split across classes
)

print(f"Train samples: {len(X_train)}")
print(f"Val samples:   {len(X_val)}")

train_dataset = PianoRollDataset(X_train, y_train)
val_dataset   = PianoRollDataset(X_val,   y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True)

Train samples: 28889
Val samples:   7223


In [ ]:
def make_cnn_block(in_ch, out_ch):
    """Single conv block: Conv → BN → ReLU → MaxPool"""
    return [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),
    ]

class GeneralCNN(nn.Module):
  """
  Configurable CNN where depth is controlled by `model_config`.

  model_config options:
    'cnn3_mlp1' - 3 conv blocks, 1 hidden MLP layer
    'cnn4_mlp1' - 4 conv blocks, 1 hidden MLP layer  (original architecture)
    'cnn5_mlp1' - 5 conv blocks, 1 hidden MLP layer
    'cnn5_mlp2' - 5 conv blocks, 2 hidden MLP layers

  Input shape: (batch, 1, 128, 468)
  After each MaxPool2d(2,2) the spatial dims halve.
  """
  # Channel progression for each CNN depth
  CHANNEL_CONFIGS = {
      'cnn3': [32, 64, 128],
      'cnn4': [32, 64, 128, 256],
      'cnn5': [32, 64, 128, 256, 512],
  }

  # Spatial dims after N max pools applied to (128, 468)
  # 128 → 64 → 32 → 16 → 8  → 4
  # 468 → 234 → 117 → 58 → 29 → 14
  SPATIAL_AFTER_POOLS = {
      3: (16, 58),
      4: (8,  29),
      5: (4,  14),
  }

  def __init__(self, num_classes, model_config='cnn4_mlp1', dropout=0.3):
      super().__init__()

      cnn_key, mlp_key = model_config.split('_')   # e.g. 'cnn4', 'mlp1'
      num_cnn = int(cnn_key[3:])                    # 3, 4, or 5
      num_mlp = int(mlp_key[3:])                    # 1 or 2

      channels = self.CHANNEL_CONFIGS[cnn_key]
      final_ch = channels[-1]
      h, w = self.SPATIAL_AFTER_POOLS[num_cnn]
      flat_size = final_ch * h * w

      # Build CNN blocks dynamically
      layers = []
      in_ch = 1
      for out_ch in channels:
          layers += make_cnn_block(in_ch, out_ch)
          in_ch = out_ch
      self.features = nn.Sequential(*layers)

      # Build MLP head dynamically
      # 1 hidden layer: flat → 512 → num_classes
      # 2 hidden layers: flat → 512 → 256 → num_classes
      mlp = [nn.Flatten(), nn.Linear(flat_size, 512), nn.ReLU(), nn.Dropout(dropout)]
      if num_mlp == 2:
          mlp += [nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout)]
          mlp += [nn.Linear(256, num_classes)]
      else:
          mlp += [nn.Linear(512, num_classes)]
      self.classifier = nn.Sequential(*mlp)

  def forward(self, x):
      x = self.features(x)
      x = self.classifier(x)
      return x

In [ ]:
class EarlyStopper:
    def __init__(self, patience=4, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_val_acc = 0

    def early_stop(self, validation_acc):
        if validation_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = validation_acc
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
learning_rates  = [1e-3, 5e-4, 3e-4, 1e-4, 1e-5]
dropout_rates   = [0.2, 0.3, 0.4]
model_configs   = ['cnn3_mlp1', 'cnn4_mlp1', 'cnn4_mlp2', 'cnn5_mlp1', 'cnn5_mlp2']

all_combinations = list(itertools.product(learning_rates, dropout_rates, model_configs))
# print(f"Total trials: {len(all_combinations)}")   # 5 × 3 × 4 = 60

results = []
OUTPUT_DIR = '/content/drive/MyDrive/cnn_model_output/hpsearch/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Default hyperparameters (your original model) ─────────────────────────────
DEFAULT_LR      = 3e-4
DEFAULT_DROPOUT = 0.3
DEFAULT_CONFIG  = 'cnn4_mlp2'

# ── Define the 12 trials as independent sweeps ────────────────────────────────
trials = (
    # Sweep 1: learning rate (dropout and config held at default)
    [{'lr': lr, 'dropout': DEFAULT_DROPOUT, 'config': DEFAULT_CONFIG} for lr in learning_rates]
    +
    # Sweep 2: dropout (lr and config held at default)
    [{'lr': DEFAULT_LR, 'dropout': do, 'config': DEFAULT_CONFIG} for do in dropout_rates]
    +
    # Sweep 3: model config (lr and dropout held at default)
    [{'lr': DEFAULT_LR, 'dropout': DEFAULT_DROPOUT, 'config': cfg} for cfg in model_configs]
)

print(f"Total trials: {len(trials)}")  # 5 + 3 + 4 = 12

results = []
OUTPUT_DIR = '/content/drive/MyDrive/cnn_model_output/hpsearch/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for trial_num, params in enumerate(trials):
    lr, dropout, config = params['lr'], params['dropout'], params['config']

    print(f"\n{'='*60}")
    print(f"Trial {trial_num+1}/{len(trials)} | lr={lr} | dropout={dropout} | config={config}")
    print(f"{'='*60}")

    model = GeneralCNN(num_classes=NUM_CLASSES, model_config=config, dropout=dropout).cuda()

    criterion     = nn.CrossEntropyLoss()
    optimizer     = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler        = GradScaler('cuda')
    scheduler     = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.7)
    early_stopper = EarlyStopper(patience=3, min_delta=0.25)

    best_val_acc = 0.0
    trial_start  = time.time()

    for epoch in range(25):
        # --- Training ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, lbls in train_loader:
            images, lbls = images.cuda(), lbls.cuda()
            optimizer.zero_grad()

            with autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, lbls)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(lbls).sum().item()
            total   += lbls.size(0)

        train_acc = 100. * correct / total

        # --- Validation ---
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, lbls in val_loader:
                images, lbls = images.cuda(), lbls.cuda()
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(lbls).sum().item()
                val_total   += lbls.size(0)

        val_acc = 100. * val_correct / val_total

        # --- Save best model for this trial ---
        if val_acc > early_stopper.best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(),
                       f'{OUTPUT_DIR}best_trial{trial_num+1}_lr{lr}_do{dropout}_{config}.pth')

        print(f"  Epoch {epoch+1:2d}/25 | Loss: {running_loss/len(train_loader):.4f} | "
              f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")

        scheduler.step()

        if early_stopper.early_stop(val_acc):
            print(f"  Early stop at epoch {epoch+1}, best val: {early_stopper.best_val_acc:.1f}%")
            break

    trial_time = (time.time() - trial_start) / 60
    print(f"  Trial complete | Best val acc: {best_val_acc:.1f}% | Time: {trial_time:.1f} min")

    results.append({
        'trial':        trial_num + 1,
        'lr':           lr,
        'dropout':      dropout,
        'config':       config,
        'best_val_acc': best_val_acc,
        'time_min':     round(trial_time, 2),
    })

Total trials: 13

Trial 1/13 | lr=0.001 | dropout=0.3 | config=cnn4_mlp2
  Epoch  1/25 | Loss: 2.4726 | Train: 17.2% | Val: 18.5%
  Epoch  2/25 | Loss: 1.9406 | Train: 25.5% | Val: 39.4%
  Epoch  3/25 | Loss: 1.8097 | Train: 32.5% | Val: 48.5%
  Epoch  4/25 | Loss: 1.7806 | Train: 34.1% | Val: 52.3%
  Epoch  5/25 | Loss: 1.7572 | Train: 34.8% | Val: 48.7%
  Epoch  6/25 | Loss: 1.7209 | Train: 36.4% | Val: 50.8%
  Epoch  7/25 | Loss: 1.7129 | Train: 37.1% | Val: 55.2%
  Epoch  8/25 | Loss: 1.7020 | Train: 38.0% | Val: 55.1%
  Epoch  9/25 | Loss: 1.6829 | Train: 39.1% | Val: 53.5%
  Epoch 10/25 | Loss: 1.6612 | Train: 40.3% | Val: 52.7%
  Early stop at epoch 10, best val: 55.2%
  Trial complete | Best val acc: 55.2% | Time: 3.1 min

Trial 2/13 | lr=0.0005 | dropout=0.3 | config=cnn4_mlp2
  Epoch  1/25 | Loss: 2.1792 | Train: 21.3% | Val: 46.9%
  Epoch  2/25 | Loss: 1.6057 | Train: 42.1% | Val: 57.5%
  Epoch  3/25 | Loss: 1.4036 | Train: 50.2% | Val: 62.4%
  Epoch  4/25 | Loss: 1.3358 | T

In [ ]:
results_df = pd.DataFrame(results).sort_values('best_val_acc', ascending=False)
print("\n" + "="*60)
print("HYPERPARAMETER SEARCH COMPLETE")
print("="*60)
print(results_df.to_string(index=False))

results_df.to_csv(f'{OUTPUT_DIR}hpsearch_results.csv', index=False)
print(f"\nResults saved to {OUTPUT_DIR}hpsearch_results.csv")

best = results_df.iloc[0]
print(f"\nBest config: lr={best['lr']} | dropout={best['dropout']} | "
      f"model={best['config']} | val_acc={best['best_val_acc']:.1f}%")



HYPERPARAMETER SEARCH COMPLETE
 trial      lr  dropout    config  best_val_acc  time_min
     4 0.00010      0.3 cnn4_mlp2     79.108404      5.33
    11 0.00030      0.3 cnn4_mlp2     78.873044      7.19
    13 0.00030      0.3 cnn5_mlp2     78.388481      4.66
     5 0.00001      0.3 cnn4_mlp2     78.319258      6.59
     6 0.00030      0.2 cnn4_mlp2     77.405510      3.45
     9 0.00030      0.3 cnn3_mlp1     77.142462      7.10
    12 0.00030      0.3 cnn5_mlp1     76.477918      4.02
     3 0.00030      0.3 cnn4_mlp2     76.173335      3.44
     7 0.00030      0.3 cnn4_mlp2     75.647238      2.83
     8 0.00030      0.4 cnn4_mlp2     75.287277      4.42
     2 0.00050      0.3 cnn4_mlp2     73.584383      3.45
    10 0.00030      0.3 cnn4_mlp1     73.418247      3.17
     1 0.00100      0.3 cnn4_mlp2     55.226360      3.12

Results saved to /content/drive/MyDrive/cnn_model_output/hpsearch/hpsearch_results.csv

Best config: lr=0.0001 | dropout=0.3 | model=cnn4_mlp2 | val_acc=79